# 05 - Debug Failed Responses

Debug notebook for investigating:
- Empty/null responses
- Failed inference (`ok=false`)
- Error messages
- Model-specific issues

In [ ]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths - all ares notebooks are in artemis_final/notebooks/ares/
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f" ARTEMIS_DIR: {ARTEMIS_DIR}")

import pandas as pd
from sqlalchemy import text
from ares.db.connection import get_engine
from ares.configs.db_config import TABLES, MODEL_NAMES

engine = get_engine()
print('Connected!')

## 1. Overview of Failed Responses

In [ ]:
# Count successful vs failed by model
query = '''
SELECT model_name, 
       COUNT(*) as total,
       SUM(CASE WHEN ok = true THEN 1 ELSE 0 END) as success,
       SUM(CASE WHEN ok = false THEN 1 ELSE 0 END) as failed,
       SUM(CASE WHEN response_raw IS NULL OR response_raw = '' THEN 1 ELSE 0 END) as empty_response
FROM vlm_responses
GROUP BY model_name
ORDER BY failed DESC
'''
df_summary = pd.read_sql(query, engine)
df_summary['fail_rate'] = (df_summary['failed'] / df_summary['total'] * 100).round(1)
df_summary

## 2. Error Messages Analysis

In [ ]:
# Get error messages for failed responses
query = '''
SELECT model_name, error_message, COUNT(*) as count
FROM vlm_responses
WHERE ok = false OR error_message IS NOT NULL
GROUP BY model_name, error_message
ORDER BY count DESC
LIMIT 20
'''
df_errors = pd.read_sql(query, engine)
print(f'Found {len(df_errors)} distinct error types')
df_errors

## 3. Empty Responses Detail

In [ ]:
# Get samples with empty responses
query = '''
SELECT r.sample_id, r.model_name, s.source_config, s.router_task,
       r.error_message, r.stop_reason, r.latency_ms
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
WHERE r.response_raw IS NULL OR r.response_raw = ''
ORDER BY r.model_name, s.source_config
'''
df_empty = pd.read_sql(query, engine)
print(f'Empty responses: {len(df_empty)}')

if len(df_empty) > 0:
    print('\nBy model:')
    print(df_empty['model_name'].value_counts())
    print('\nBy config:')
    print(df_empty['source_config'].value_counts().head(10))
    print('\nSample empty responses:')
    display(df_empty.head(10))

## 4. Specific Model Debug: Qwen3 & DeepSeek OCR

In [ ]:
# Debug specific models
debug_models = ['qwen3_vl_8b_thinking', 'deepseek_ocr']

for model in debug_models:
    print('='*60)
    print(f'MODEL: {model}')
    print('='*60)
    
    # Overall stats
    query = f'''
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN ok THEN 1 ELSE 0 END) as success,
        SUM(CASE WHEN response_raw IS NULL OR response_raw = '' THEN 1 ELSE 0 END) as empty,
        AVG(latency_ms) as avg_latency,
        AVG(input_tokens) as avg_input_tokens,
        AVG(output_tokens) as avg_output_tokens
    FROM vlm_responses WHERE model_name = '{model}'
    '''
    stats = pd.read_sql(query, engine).iloc[0]
    print(f'Total: {stats["total"]}, Success: {stats["success"]}, Empty: {stats["empty"]}')
    print(f'Avg Latency: {stats["avg_latency"]:.0f}ms')
    print(f'Avg Tokens: {stats["avg_input_tokens"]:.0f} in, {stats["avg_output_tokens"]:.0f} out')
    
    # Error breakdown
    query = f'''
    SELECT error_message, stop_reason, COUNT(*) as count
    FROM vlm_responses
    WHERE model_name = '{model}' AND (ok = false OR error_message IS NOT NULL)
    GROUP BY error_message, stop_reason
    ORDER BY count DESC
    LIMIT 5
    '''
    errors = pd.read_sql(query, engine)
    if len(errors) > 0:
        print('\nError breakdown:')
        display(errors)
    
    # Sample failed responses
    query = f'''
    SELECT r.sample_id, s.source_config, s.prompt_text, r.error_message, r.stop_reason
    FROM vlm_responses r
    JOIN vlm_samples s ON r.sample_id = s.sample_id
    WHERE r.model_name = '{model}' AND (r.ok = false OR r.response_raw IS NULL OR r.response_raw = '')
    LIMIT 3
    '''
    samples = pd.read_sql(query, engine)
    if len(samples) > 0:
        print('\nSample failed requests:')
        for idx, row in samples.iterrows():
            print(f"  Sample: {row['sample_id']}")
            print(f"  Config: {row['source_config']}")
            print(f"  Prompt: {row['prompt_text'][:100]}...")
            print(f"  Error: {row['error_message']}")
            print(f"  Stop: {row['stop_reason']}")
            print()

## 5. Compare Working vs Failed Samples

In [ ]:
# Find samples where SOME models worked but others failed
query = '''
SELECT sample_id,
       SUM(CASE WHEN ok THEN 1 ELSE 0 END) as success_count,
       array_agg(CASE WHEN ok THEN model_name ELSE NULL END) as success_models,
       array_agg(CASE WHEN NOT ok THEN model_name ELSE NULL END) as fail_models
FROM vlm_responses
GROUP BY sample_id
HAVING SUM(CASE WHEN ok THEN 1 ELSE 0 END) > 0 
   AND SUM(CASE WHEN NOT ok THEN 1 ELSE 0 END) > 0
LIMIT 10
'''
df_partial = pd.read_sql(query, engine)
print(f'Samples with partial success: {len(df_partial)}')
if len(df_partial) > 0:
    display(df_partial)

## 6. Stop Reason Analysis

In [ ]:
# Analyze stop reasons
query = '''
SELECT model_name, stop_reason, COUNT(*) as count
FROM vlm_responses
GROUP BY model_name, stop_reason
ORDER BY model_name, count DESC
'''
df_stop = pd.read_sql(query, engine)
pivot = df_stop.pivot(index='model_name', columns='stop_reason', values='count').fillna(0).astype(int)
print('Stop reasons by model:')
pivot

## 7. Check GPU Metrics Availability

In [ ]:
# Check if GPU metrics were captured
query = '''
SELECT model_name,
       COUNT(*) as total,
       SUM(CASE WHEN gpu_name IS NOT NULL THEN 1 ELSE 0 END) as has_gpu_name,
       SUM(CASE WHEN gpu_util_percent IS NOT NULL THEN 1 ELSE 0 END) as has_gpu_util,
       AVG(gpu_util_percent) as avg_util,
       AVG(gpu_mem_used_mb) as avg_mem_mb
FROM vlm_responses
GROUP BY model_name
'''
df_gpu = pd.read_sql(query, engine)
df_gpu['gpu_capture_rate'] = (df_gpu['has_gpu_name'] / df_gpu['total'] * 100).round(1)
df_gpu

## 8. Retry Failed Samples (Manual)

In [ ]:
# Get list of sample_ids that need retry for a specific model
model_to_retry = 'qwen3_vl_8b_thinking'

query = f'''
SELECT sample_id
FROM vlm_responses
WHERE model_name = '{model_to_retry}'
  AND (ok = false OR response_raw IS NULL OR response_raw = '')
'''
retry_ids = pd.read_sql(query, engine)['sample_id'].tolist()
print(f'Samples to retry for {model_to_retry}: {len(retry_ids)}')
if retry_ids:
    print(f'Sample IDs: {retry_ids[:10]}...')

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict
@dataclass
class InferenceConfig:
    run_id: str = field(default_factory=lambda: f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    models_yaml: str = '../configs/models.yaml'
    gpu_endpoints: Dict[str, str] = field(default_factory=lambda: {
        'deepseek_ocr': 'http://localhost:9002/metrics',
        'qwen2_5_vl_3b': 'http://localhost:9001/metrics',
        'qwen2_5_vl_7b': 'http://localhost:9001/metrics',
        'qwen3_vl_8b_thinking': 'http://localhost:9000/metrics',
        'gemma_3_27b': 'http://localhost:9000/metrics',
    })
    samples_per_config: int = 50
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    temperature: float = 0.0
    max_tokens: int = 512
    top_p: float = 1.0
    
    # Parallelization
    max_config_workers: int = 4       # Parallel configs
    max_model_workers: int = 5        # Parallel models per sample
    batch_insert_size: int = 50       # Insert every N samples

config = InferenceConfig()
print(f'Run ID: {config.run_id}')
print(f'Workers: {config.max_config_workers} configs x {config.max_model_workers} models')

In [ ]:
# Replay a failed qwen3_vl_8b_thinking request through the inference API
from ares.inference_api_call.client import WhichVLMClient

config_path = '../configs/models.yaml'
print('Loading inference client from', config_path)
client = WhichVLMClient.from_yaml(config_path)

model_to_retry = 'qwen3_vl_8b_thinking'
sample_query = text('''
SELECT r.sample_id
FROM vlm_responses r
WHERE r.model_name = :model
  AND (r.ok = false OR r.response_raw IS NULL OR r.response_raw = '')
ORDER BY r.updated_at DESC
LIMIT 1
''')

In [ ]:

sample_ids = pd.read_sql(sample_query, engine, params={'model': model_to_retry})
if sample_ids.empty:
    print(f'No failed {model_to_retry} rows found to replay.')
else:
    sample_id = sample_ids.iloc[0]['sample_id']
    detail_query = text('''
SELECT s.sample_id, s.prompt_text, s.prompt_formatted, s.system_prompt,
       i.image_bytes
FROM vlm_samples s
LEFT JOIN vlm_images i ON s.image_id = i.image_id
WHERE s.sample_id = :sample_id
''')
    
    print(sample_id)
    detail_df = pd.read_sql(detail_query, engine, params={'sample_id': sample_id})
    detail = detail_df.iloc[0]
    prompt_text = detail['prompt_formatted'] if pd.notna(detail['prompt_formatted']) else detail['prompt_text']
    prompt_text = (prompt_text or '').strip()
    
    system_prompt = detail['system_prompt'] if pd.notna(detail['system_prompt']) else None
    image_bytes = detail['image_bytes']
    if isinstance(image_bytes, memoryview):
        image_bytes = image_bytes.tobytes()
    print(f"Replaying Sample: {sample_id} | prompt length: {len(prompt_text)} chars")
    print(f"Prompt Text:\n{prompt_text}\n")
    # img = load_image_from_db(sample['image_id'].iloc[0])
    
    vlm_resp = client.vlm.run_image(
        image=image_bytes,
        text=prompt_text,
        models=model_to_retry,
        system=system_prompt,
    )
    resp = vlm_resp.get(model_to_retry)
    print(resp)
    # if resp is None:
    #     print('No response returned for', model_to_retry)
    # else:
    #     print('Response text:')
    #     print(resp['response_text'])
    #     print('Tokens (in/out):', resp['input_tokens'], resp['output_tokens'])
    #     print('Latency (ms):', resp['latency_ms'])
    #     print('Est. cost (USD):', resp['est_cost'])
